# Steerling-8B: Finding Concepts to Steer

Before amplifying or suppressing a concept you need its **concept ID**. This notebook shows how to search the concept catalog by meaning and confirm a concept does what its label says by inspecting the model internal concepts.

Two tools work together. `ConceptCatalog` searches the names and descriptions in `assets/concepts/concept_labels.parquet`. `SteerlingGenerator.concept_top_tokens` shows the vocabulary a concept actually promotes in the loaded weights, which is the reliable check: concept IDs are specific to a checkpoint, so a label from another table can point at the wrong concept.

## 1. Load the catalog

The catalog reads the bundled concept table. No model needed for search.

In [ ]:
from steerling import ConceptCatalog

catalog = ConceptCatalog.load()
print(len(catalog), "concepts")

## 2. Search by meaning, input the keyword into `catalog.search()`

Search covers both the name and the description, matching whole words. Pass a few related terms. By default only **steerable known** concepts are returned, which is the set you can steer.

In [ ]:
for c in catalog.search("Book", limit=8):
    print(f"{c.concept_id:6d}  {c.name!r}  (group: {c.group_name})")

## 3. Read a concept

Once you have an ID, look at its full record. The category flags mark sensitive concepts: `is_demographic`, `is_alignment`, and `is_tone`.

In [ ]:
CONCEPT_ID = 20353  # replace with an ID from your search

c = catalog.get(CONCEPT_ID)
print("name:       ", c.name)
print("group:      ", c.group_name)
print("steerable:  ", c.is_steerable)
print("flags:      ", f"tone={c.is_tone} alignment={c.is_alignment} demographic={c.is_demographic}")
print("description:\n", c.description)

## 4. Confirm against the weights of `Base` mode. Replace with `Instruct` model if needed

The label is the intended meaning. To see what the concept actually does in this checkpoint, load the model and read the tokens it most promotes. If these match the label, the ID is correct. If they look unrelated, the ID is from a different concept table.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator

model = AutoModel.from_pretrained("guidelabs/steerling-8b", trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained("guidelabs/steerling-8b", trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer)

In [ ]:
for token, score in generator.concept_top_tokens(CONCEPT_ID, k=15):
    print(f"{score:6.3f}  {token!r}")